# LiDAR Tree & Building Detection on an ArcGIS Map

Open-source Python processes the LiDAR point cloud (builds the Canopy Height Model and detects **trees** and **buildings**), then the **ArcGIS API for Python** displays the results on an interactive **ArcGIS** map over a real basemap.

> Uses an anonymous ArcGIS connection (no login, no publishing). Sample tile is placed near Albany, NY (UTM Zone 18N).

In [1]:
# 1) Run the LiDAR pipeline (open-source: laspy + scipy + rasterio)
from lidar_tree_building_detector import process, world_features

dem, dsm, chm, transform, buildings, trees = process('data/sample_lidar.las', cell=1.0)
tree_pts, bldg_polys, wkid = world_features(trees, buildings, transform)

print(f'Detected {len(tree_pts)} trees and {len(bldg_polys)} building(s).')
print(f'Tallest tree: {max(t["height_m"] for t in tree_pts):.1f} m')

Detected 262 trees and 1 building(s).
Tallest tree: 21.7 m


In [2]:
# 2) Connect to ArcGIS (anonymous) and build spatial layers from the detections
from arcgis.gis import GIS
from arcgis.features import FeatureSet, Feature
from arcgis.geometry import Point, Polygon

gis = GIS()   # anonymous, no login

# trees -> point features
tree_feats = [
    Feature(geometry=Point({'x': t['x'], 'y': t['y'], 'spatialReference': {'wkid': wkid}}),
            attributes={'height_m': t['height_m']})
    for t in tree_pts
]
tree_fset = FeatureSet(tree_feats, geometry_type='esriGeometryPoint',
                       spatial_reference={'wkid': wkid})

# buildings -> polygon features
bldg_feats = [
    Feature(geometry=Polygon({'rings': [b['ring']], 'spatialReference': {'wkid': wkid}}),
            attributes={'area_m2': b['area_m2'], 'height_m': b['height_m']})
    for b in bldg_polys
]
bldg_fset = FeatureSet(bldg_feats, geometry_type='esriGeometryPolygon',
                       spatial_reference={'wkid': wkid})
print('Built', len(tree_feats), 'tree points and', len(bldg_feats), 'building polygon(s).')

Built 262 tree points and 1 building polygon(s).


In [6]:
# 3) Display on an ArcGIS map, zoomed to the LiDAR tile
m = gis.map('Albany, New York')
m.zoom = 16

# add detections as layers (newer arcgis API uses .content.add)
m.content.add(bldg_fset)
m.content.add(tree_fset)
m.zoom = 18
m

Map()

Green points are detected treetops; the red polygon is the detected building footprint, shown on a real ArcGIS basemap near Albany. Take a screenshot of the map for your post.

**Note:** the LiDAR analysis (CHM + detection) is done with open-source Python; the ArcGIS API for Python handles the spatial display. Swap in a real LAS tile with its own coordinate system and it maps in the right place automatically.